# DPO evaluation only (Google Colab T4)

This notebook restores and validates the five completed training stages, then evaluates the six checkpoints on the deterministic 200-example validation subset. **No training is required or performed.** Do not run the original training notebook as part of this workflow.

Select a Tesla T4 GPU before running. Add a write-enabled `HF_TOKEN` to Colab Secrets and enable notebook access. Run the five code cells in order. The evaluation uses the existing 256-token generation cap, saves to a separate `evaluation-200` namespace, resumes completed checkpoint evaluations, and uploads results to the private Hub repository. It does not access the reserved final-test partition.

The notebook requires GitHub `main` to contain baseline commit `71989b12bb5c3f9b1bf41817c16c7ee2e291750c` or a descendant. The runtime config is written under `/content`; repository training configs and checkpoints are not edited.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Adhilmohdkt/dpo-from-scratch.git"
BRANCH = "main"
MINIMUM_BASELINE = "71989b12bb5c3f9b1bf41817c16c7ee2e291750c"
PROJECT_DIR = Path("/content/dpo-from-scratch-evaluation")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError(f"Project path exists but is not a Git checkout: {PROJECT_DIR}")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)

remote_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", f"origin/{BRANCH}"], text=True
).strip()
ancestor = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "merge-base", "--is-ancestor", MINIMUM_BASELINE, f"origin/{BRANCH}"],
    check=False,
)
if ancestor.returncode != 0:
    raise RuntimeError(
        f"Required evaluation baseline {MINIMUM_BASELINE} is missing from or is not an ancestor of GitHub {BRANCH} ({remote_commit})."
    )
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "--force", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
checked_out_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()
checked_out_branch = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "symbolic-ref", "--short", "HEAD"], text=True
).strip()
required_files = [
    "requirements-colab.txt", "configs/scaled_t4.yaml", "configs/scaled_t4_eval_200.yaml",
    "scripts/run_evaluation_only.py", "scripts/aggregate_results.py", "src/evaluate.py",
    "docs/EVALUATION_ONLY.md",
]
missing = [name for name in required_files if not (PROJECT_DIR / name).is_file()]
if checked_out_branch != BRANCH or checked_out_commit != remote_commit or missing:
    raise RuntimeError(
        f"Invalid checkout: expected {BRANCH}@{remote_commit}; got {checked_out_branch}@{checked_out_commit}; missing files={missing}"
    )

import os
os.chdir(PROJECT_DIR)
print({"branch": checked_out_branch, "commit": checked_out_commit, "minimum_baseline_is_ancestor": True, "required_files_present": True})


In [ ]:
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Colab, select Runtime > Change runtime type > T4 GPU, then reconnect.")
gpu_name = torch.cuda.get_device_name(0)
if "T4" not in gpu_name:
    raise RuntimeError(f"This workflow expects a Tesla T4; detected {gpu_name!r}.")
managed_torch = torch.__version__.split("+")[0]
print({"python": sys.version.split()[0], "torch": torch.__version__, "cuda_available": True, "gpu": gpu_name})

# requirements-colab.txt intentionally preserves Colab's managed Torch/CUDA build.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
from importlib.metadata import version
installed_torch = version("torch").split("+")[0]
if installed_torch != managed_torch or torch.__version__.split("+")[0] != managed_torch:
    raise RuntimeError(f"Dependency setup changed Colab-managed PyTorch ({managed_torch} -> {installed_torch}); reconnect to a fresh runtime.")
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("CUDA/T4 is unavailable after dependency installation.")
import transformers, peft, datasets, trl, accelerate, yaml
print({"transformers": transformers.__version__, "peft": peft.__version__, "datasets": datasets.__version__, "trl": trl.__version__, "accelerate": accelerate.__version__, "torch_preserved": torch.__version__})


In [ ]:
import os
import yaml
from google.colab import userdata
from huggingface_hub import HfApi

BASE_CONFIG_PATH = Path("configs/scaled_t4.yaml")
RUNTIME_CONFIG_PATH = Path("/content/scaled_t4_hub_run.yaml")
HUB_REPO_ID = "adhill/dpo-experiment-checkpoint"
config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
expected = {
    "seed": 42,
    "training_pairs": 512,
    "betas": [0.1, 0.5],
    "selection_pairs": 128,
    "selection_beta": 0.1,
    "dataset_revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
    "validation_pairs": 500,
    "final_test_pairs": 500,
    "max_new_tokens": 256,
}
actual = {
    "seed": config["seed"],
    "training_pairs": config["data"]["max_train_samples"],
    "betas": config["experiment"]["beta_values"],
    "selection_pairs": config["experiment"]["selection_subset_size"],
    "selection_beta": config["experiment"]["selection_beta"],
    "dataset_revision": config["data"]["revision"],
    "validation_pairs": config["data"]["validation_eval_samples"],
    "final_test_pairs": config["data"]["max_eval_samples"],
    "max_new_tokens": config["evaluation"]["max_new_tokens"],
}
if actual != expected or config["experiment"]["seeds"] != [42]:
    raise RuntimeError(f"Original experiment config differs from the completed training plan: {actual}")
config["hub"]["repo_id"] = HUB_REPO_ID
if config["hub"].get("enabled") is not True or config["hub"].get("private") is not True:
    raise RuntimeError("Private Hugging Face Hub persistence must remain enabled.")
RUNTIME_CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

try:
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Could not read HF_TOKEN from Colab Secrets. Add it and enable notebook access.") from None
if not _hf_token:
    raise RuntimeError("HF_TOKEN is missing or empty. Add a write-enabled token to Colab Secrets.")
os.environ["HF_TOKEN"] = _hf_token
api = HfApi(token=_hf_token)
del _hf_token
try:
    authenticated = bool(api.whoami())
    repo_info = api.repo_info(repo_id=HUB_REPO_ID, repo_type="model")
except Exception as exc:
    api = None
    raise RuntimeError(f"Hugging Face authentication or repository access failed ({type(exc).__name__}).") from None
api = None
if not authenticated or getattr(repo_info, "private", None) is not True:
    raise RuntimeError(f"Expected authenticated access to private repository {HUB_REPO_ID}.")
print({"runtime_config": str(RUNTIME_CONFIG_PATH), "experiment_config_matches": True, "hub_authentication": "passed", "target_repo_private": True})


In [ ]:
!python -m scripts.run_evaluation_only \
  --config /content/scaled_t4_hub_run.yaml \
  --repo-id adhill/dpo-experiment-checkpoint \
  --work-dir /content/dpo-evaluation-only \
  --evaluation-plan configs/scaled_t4_eval_200.yaml


In [ ]:
import csv
from pathlib import Path
import yaml

plan = yaml.safe_load(Path("configs/scaled_t4_eval_200.yaml").read_text(encoding="utf-8"))
summary_path = Path(plan["output_dir"]) / "seed-42" / "validation" / "checkpoint_summary.csv"
if not summary_path.is_file():
    raise FileNotFoundError(f"Evaluation summary is missing: {summary_path}")
with summary_path.open(encoding="utf-8", newline="") as stream:
    summary_rows = list(csv.DictReader(stream))
expected_labels = {"base", "sft", "dpo_beta_0.1", "dpo_beta_0.5", "dpo_margin", "dpo_random"}
if {row.get("checkpoint") for row in summary_rows} != expected_labels:
    raise RuntimeError("Evaluation summary does not contain all six expected checkpoints.")
print("Local summary:", summary_path)
print("Hub evaluation path: scaled-t4-512-validation-200/seed-42/evaluation/validation")
for row in summary_rows:
    print(row)
